# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adeenafatima0/ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

Going with Random Forest. My baseline used a rigid rule (visible + declining = high score), but the top-20 review showed a real weakness: every top pick tied at the same score, since the rule can only ask a couple of yes/no questions. A Random Forest can combine many weaker signals with different weights instead of hard cutoffs, which should let it rank pages within that "tied" group instead of treating them all as equally urgent. I also already saw in notebook 01 that Random Forest clearly outperformed a hand-rule (0.740 vs 0.240 Precision@50), so it's a proven fit for this exact kind of problem.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

Using a client-grouped split, not a random split. Pages from the same client likely share patterns (same site structure, same content strategy), so if pages from one client end up in both train and test, the model could just be memorizing that client rather than learning a generalizable pattern. A client-holdout split means whole clients are set aside for testing, so the model is judged on clients it's never seen — a more honest test of whether it actually learned something useful.

In [6]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/adeenafatima0/ml-internship"
REPO_DIR = "ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Client-grouped split: whole clients go to either train or test, never both
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

print("Train rows:", len(train_df), "| Test rows:", len(test_df))
print("Train clients:", train_df["client_id"].nunique(), "| Test clients:", test_df["client_id"].nunique())
print("Overlap check (should be 0):", len(set(train_df["client_id"]) & set(test_df["client_id"])))

Train rows: 22885 | Test rows: 7115
Train clients: 24 | Test clients: 8
Overlap check (should be 0): 0


## 3. Train + compare vs my baseline

Training a Random Forest on the same train/test client split, then comparing Precision@50 against my Week-4 baseline rule — both scored on the exact same test set.

In [7]:
from sklearn.ensemble import RandomForestClassifier

features = ["impressions_90d", "days_since_last_update", "avg_position",
            "ctr", "word_count", "content_age_days"]

X_train = train_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_train = train_df["is_declining_label"]
X_test = test_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_test = test_df["is_declining_label"]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

# Recreate the baseline rule score on the test set, for a fair comparison
test_df["is_visible"] = test_df["impressions_90d"] >= 500
test_df["is_stale"] = test_df["days_since_last_update"] >= 180
baseline_scores = (
    test_df["is_visible"].astype(int) * 2
    + test_df["is_declining_label"].astype(int) * 3
    + test_df["is_stale"].astype(int) * 1
)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y_test_arr = y_test.values
results = []
for k in (20, 50):
    base_p = precision_at_k(baseline_scores.values, y_test_arr, k)
    rf_p = precision_at_k(rf_scores, y_test_arr, k)
    results.append({"k": k, "baseline_precision": round(base_p, 3), "rf_precision": round(rf_p, 3)})

results_df = pd.DataFrame(results)
results_df
# Corrected baseline: score using ONLY observable signals, not the label itself
baseline_scores_fixed = (
    test_df["is_visible"].astype(int) * 2
    + test_df["is_stale"].astype(int) * 1
)

results = []
for k in (20, 50):
    base_p = precision_at_k(baseline_scores_fixed.values, y_test_arr, k)
    rf_p = precision_at_k(rf_scores, y_test_arr, k)
    results.append({"k": k, "baseline_precision": round(base_p, 3), "rf_precision": round(rf_p, 3)})

results_df = pd.DataFrame(results)
results_df

,k,baseline_precision,rf_precision
0,20,0.70,0.55
1,50,0.64,0.52


## 4. Errors and interpretation

Surprising result: my baseline rule actually beat the Random Forest on this client-holdout split (0.70 vs 0.55 Precision@20, 0.64 vs 0.52 Precision@50). This is worth reporting honestly rather than hiding, since it's a real finding, not a bug.

Likely reasons:
- The test set is small (7,115 rows, only 8 clients), so results may be sensitive to which specific clients landed in the holdout group.
- The baseline isn't a weak rule — it leans on visibility, which I confirmed earlier has a genuinely meaningful relationship with decline (59.6% vs 47.5%). So it's already capturing real signal, not guessing blindly.
- The Random Forest may be overfitting to patterns in the training clients that don't transfer well to entirely new clients, especially with a relatively small, noisy dataset.

What this tells me: a more complex model isn't automatically better, especially with limited data and a strict client-holdout test. Next steps worth trying: tune the Random Forest's depth/complexity down, try Logistic Regression as a simpler alternative, or get a larger test set from the full warehouse data instead of the 30K-row starter sample.
Feature importance shows the model leans most on impressions_90d (33%), avg_position (23%), and content_age_days (20%) — with days_since_last_update barely used at all (5%). This actually lines up with my earlier signal check in Week 4, where staleness turned out to be a weak, backwards signal for decline. It's reassuring that the model independently arrived at a similar conclusion using real data, rather than me just guessing it by hand.

In [8]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

impressions_90d           0.328400
avg_position              0.230381
content_age_days          0.197370
word_count                0.128825
ctr                       0.066138
days_since_last_update    0.048886
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.